In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

In [ ]:

!pip install opendatasets --upgrade --quiet
import opendatasets as od

od.download("https://www.kaggle.com/datasets/ealaxi/paysim1")

In [ ]:

df = pd.read_csv("paysim1/PS_20174392719_1491204439457_log.csv")

print("Dataset Shape (Rows, Columns):", df.shape)

In [ ]:
print("--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- CLASS DISTRIBUTION (isFraud) ---")
print(df['isFraud'].value_counts())

In [ ]:
#  filtered table with only TRANSFER and CASH_OUT
df_clean = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# remaining row count
print("Remaining rows:", len(df_clean))

In [ ]:
df_clean.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True, errors='ignore')
# Calculate the exact mathematical error in the sender's account
df_clean['errorBalanceOrig'] = df_clean['newbalanceOrig'] + df_clean['amount'] - df_clean['oldbalanceOrg']

# Calculate the exact mathematical error in the receiver's account
df_clean['errorBalanceDest'] = df_clean['oldbalanceDest'] + df_clean['amount'] - df_clean['newbalanceDest']
df_clean.shape

In [ ]:
# CELL 7: Translate text categories into numbers
encoder = LabelEncoder()
df_clean['type'] = encoder.fit_transform(df_clean['type'])

# Print out the data types of every column to prove zero text remains
print("--- COLUMN DATA TYPES ---")
print(df_clean.dtypes)
df_clean.head()

In [ ]:
# Define features (X) and target (y)
X = df_clean.drop('isFraud', axis=1)
y = df_clean['isFraud']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# CELL 9: Balance the Training Set using SMOTE

print("--- BEFORE SMOTE (Training Set) ---")
print(y_train.value_counts())

# 1. Initialize the SMOTE tool
smote = SMOTE( sampling_strategy=0.1, random_state=42)

# 2. Generate synthetic fraud examples ONLY in the training set
print("\nRunning SMOTE... (This might take 30-60 seconds on 2.2 million rows)...")
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\n--- AFTER SMOTE (Balanced Training Set) ---")
print(y_train_balanced.value_counts())

In [ ]:
# CELL 10: Train the Random Forest Algorithm on the Balanced Data
from sklearn.ensemble import RandomForestClassifier
import time
import joblib # Import joblib for saving and loading models

print("Initializing the 50 Decision Trees...")
# 1. Set up the Random Forest with our optimized rules
model = RandomForestClassifier(
    n_estimators=200,      # 50 trees in the forest
    max_depth=10,         # Prevent overfitting
    n_jobs=-1,            # Use all Colab CPU cores for speed!
    random_state=42       # Ensure consistent results
)

print("Training started on 3.86 million rows! (This usually takes 1 to 2 minutes)...")
start_time = time.time()

# 2. Train the model (The AI is learning right now!)
model.fit(X_train_balanced, y_train_balanced)

end_time = time.time()
print(f"\n--- TRAINING COMPLETE! ---")
print(f"Time taken: {round(end_time - start_time, 2)} seconds")
print("Your Random Forest model is officially trained and ready to hunt fraudsters!")

# Export Production Model
print("\nExporting production model...")

joblib.dump(model, "fraud_model.joblib")
joblib.dump(encoder, "label_encoder.joblib")
joblib.dump(X.columns.tolist(), "feature_columns.joblib")

print("✅ fraud_model.joblib")
print("✅ label_encoder.joblib")
print("✅ feature_columns.joblib")
print("\nProduction model exported successfully!")


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

def evaluate_model(model, X_eval, y_eval):
    # Predictions
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    print(f"Evaluating rows: {len(y_eval)}\n")

    print(f"Precision : {precision_score(y_eval, y_pred):.6f}")
    print(f"Recall    : {recall_score(y_eval, y_pred):.6f}")
    print(f"F1 Score  : {f1_score(y_eval, y_pred):.6f}")
    print(f"ROC-AUC   : {roc_auc_score(y_eval, y_prob):.6f}")
    print(f"PR-AUC    : {average_precision_score(y_eval, y_prob):.6f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_eval,
        y_pred,
        target_names=["Legitimate", "Fraud"]
    ))

    # FP / FN count
    cm = confusion_matrix(y_eval, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("\nFalse Positives :", fp)
    print("False Negatives :", fn)

    return {
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
        "roc_auc": roc_auc_score(y_eval, y_prob),
        "pr_auc": average_precision_score(y_eval, y_prob),
        "false_positives": fp,
        "false_negatives": fn
    }

    results = evaluate_model(
    model,
    X_test,
    y_test
)